# Fase 0: fundamentos de redes neurais com NumPy
**Objetivo:** Compreender o funcionamento dos algoritmos fundamentais de redes neurais.

**Conceitos-chave:** regressão, gradiente descendente, neurônio, função de ativação, *backpropagation*.

### Sumário
*Etapa 0.1: Gradiente Descendente*

*Etapa 0.2: MLP do zero*

**Etapa 0.3: *Backpropagation***

*Etapa 0.4: Autograd simples*

*Mini-projeto 0: A curva de luz de uma supernova Ia*

In [1]:
import numpy as np

np.random.seed(4242) # Garante que as células resultem nos mesmos valores se executadas em ordem

# 0.3 — *Backpropagation*

O conceito de *backpropagation* é fundamental para *machine learning*. É esse mecanismo que permite uma rede neural "aprender" os padrões que desejamos entender e explorar.

## 0.3.1 O gradiente descendente e a rede neural

Na etapa $0.2$, vimos como implementar uma MLP simples de duas camadas e como propagar as informações da entrada para a saída da rede. No entanto, no estado em que a rede implementada está, ela faz simplesmente isso: recebe uma entrada e produz uma saída. Isso obviamente não é útil para o nosso problema de regressão, pois a rede é *estática*, isto é, os pesos e *bias* não mudam e, consequentemente, o padrão que a rede executa é fixo.

Para isso devemos implementar o algoritmo do gradiente descendente. Escolheremos uma função de perda que deverá ser minimizada pela rede neural. Diferente do primeiro exemplo na etapa $0.1$, ao usar redes neurais, *não supomos* uma forma para a função que queremos aproximar. A ideia é que os valores dos pesos e dos *bias* sejam alterados de modo que a função de perda seja minimizada e, consequentemente, sejam valores "ideais" para reproduzir os dados.

Primeiro, vamos implementar a função de perda na nossa classe `MLP`.

In [2]:
class Camada:
    ''' Define uma camada de neurônios.
    
    Atributos:
        n_entrada: número de entradas recebidas por cada neurônio da camada.
        n_saida: número de neurônios desta camada.        
    '''
    # Inicialização de uma camada
    def __init__(self, n_entrada, n_saida):
        self.n_entrada = n_entrada
        self.n_saida = n_saida

        self.pesos = np.random.randn(n_entrada, n_saida) # Gera a matriz de pesos cuja dimensão é n_entrada x n_saida
        self.bias = np.random.randn(1, n_saida) # Gera a matriz de bias cuja dimensão é 1 x n_saida

    # Vamos definir um método `forward` para executar o forward propagation
    def forward(self, entrada):
        ''' Executa o foward propagation pela camada.
        Parâmetros:
            entrada: valores que serão processados pelos neurônios da camada.
        '''
        M1 = np.dot(entrada, self.pesos) # Realiza a multiplicação da matriz de entrada pela matriz de pesos da camada
        M2 = M1 + self.bias # Soma a matriz de bias a todas as linhas de M1
        return M2

class MLP:
    ''' Define um multilayer perceptron de duas camadas.
    Atributos:
        camada1: a primeira camada do MLP.
        camada_saida: a camada de saída do MLP.

    Parâmetros:
        dim_n1: lista ou tupla contendo as dimensões da primeira camada na forma (n_entrada, n_saida).
        dim_n1: lista ou tupla contendo as dimensões da camada de saída na forma (n_entrada, n_saida).
    '''

    def __init__(self, dim_n1, dim_n2):
        self.camada1 = Camada(dim_n1[0], dim_n1[1])
        self.camada_saida = Camada(dim_n2[0], dim_n2[1])
        self.H1 = 0.0 # Atributo para salvarmos a saída da camada 1
        self.z = 0.0 # Atributo para salvarmos a saída da ReLU

    # Definindo a função ReLU para ser usada como função de ativação
    def relu(self, entrada):
        zeros = np.zeros_like(entrada)
        self.z = np.maximum(zeros, entrada)
        return self.z

    # Definindo o método para executar o forward propagation pela rede
    def forward(self, entrada):
        H1 = self.camada1.forward(entrada) # Passa para a camada 1
        self.H1 = H1
        z = self.relu(H1) # Passa a saída da camada 1 pela função de ativação
        y = self.camada_saida.forward(z) # Passa a saída da função de ativação para a camada de saída
        return y

    # Definindo a função de perda MSE. Parâmetros: Y: dados observados; y: dados previstos
    def MSE(self, Y, y):
        erro = Y - y
        return np.mean(erro**2)

A próxima etapa é implementar as derivadas da função de perda em relação aos parâmetros do modelo. Como dito, os parâmetros, nesse caso, são os pesos e os *bias* da rede. Então, precisamos calcular
$$
\frac{\partial L}{\partial \mathbf{W}_{1}}, \quad \frac{\partial L}{\partial \mathbf{b}_{1}}, \quad \frac{\partial L}{\partial \mathbf{W}_{2}}, \quad \frac{\partial L}{\partial \mathbf{b}_{2}}.
$$
Como a rede neural possui diferentes camadas e a função de ativação, para calcularmos essas derivadas deveremos empregar a **regra da cadeia** consecutivamente. Sendo a função de perda dada por
$$
L = \frac{1}{N}\sum_{i=1}^{N}(y_{i}-\hat{y}_{i})^{2},
$$
e que $\hat{y}$ é a saída do nosso modelo, sabemos que todas as derivadas acima dependerão de
$$
\frac{\partial L}{\partial \hat{y}_{k}}.
$$
Calculando essa derivada, ficamos com
$$
\frac{\partial L}{\partial \hat{y}_{k}} = -\frac{2}{N}(y_{k} - \hat{y}_{k}).
$$
A função de perda deve ser derivada em relação a cada elemento da matriz dos dados previstos e, por isso, o somatório desaparece. No entanto, podemos escrever de forma mais direta utilizando a notação matricial. Além disso, já que estamos lidando com matrizes, essa notação também facilitará os cálculos. Assim,
$$
\frac{\partial L}{\partial \mathbf{\hat{y}}} = -\frac{2}{N}(\mathbf{y} - \mathbf{\hat{y}}).
$$

O fluxo de trabalho da nossa rede é
$$
\mathbf{x} \quad \longrightarrow \quad N_{1}: \quad \mathbf{x}\mathbf{W}_{1} + \mathbf{b}_{1} \quad \longrightarrow \quad \mathbf{H}_{1} \quad \longrightarrow \quad \mathbf{z} = \max(0, \mathbf{H}_{1}) \quad \longrightarrow \quad N_{2}: \quad \mathbf{z}\mathbf{W}_{2} + \mathbf{b}_{2} \quad \longrightarrow \quad \mathbf{\hat{y}}.
$$
Então, as derivadas vão "acumulando-se" conforme percorremos a rede da saída para a entrada. Vamos calcular as derivadas e ver na prática. A ideia é sempre calcular de trás para frente, ou seja, da saída para a entrada. Comecemos pela derivada de $L$ em relação a $\mathbf{b}_{2}$:
$$
\frac{\partial L}{\partial \mathbf{b}_{2}} = \frac{\partial L}{\partial \mathbf{\hat{y}}} \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{b}_{2}}.
$$
Como
$$
\mathbf{\hat{y}} = \mathbf{z}\mathbf{W}_{2} + \mathbf{b}_{2} \longrightarrow \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{b}_{2}} = \mathbf{1},
$$
em que $\mathbf{1}$ é uma matriz de dimensão $5\times 1$.
e, então,
$$
\frac{\partial L}{\partial \mathbf{b}_{2}} = -\frac{2}{N}(\mathbf{y} - \mathbf{\hat{y}}) \mathbf{1}.
$$
Um detalhe **importante**: a função de perda é um escalar que depende de matrizes, então suas derivadas em relação a essas matrizes deve ter a *mesma* dimensão que as tais matrizes. Para que isso aconteça na expressão acima (e para que seja possível realizar a multiplicação matricial de fato), precisamos ajeitar a expressão. Temos uma matriz $5\times 1$ multiplicando uma matriz também $5\times 1$. Essa multiplicação não é válida e, para ajeitarmos ela de tal modo queo resultado tenha as dimensões de $\mathbf{b}_{2}$, precisamos reescrevê-la como
$$
\frac{\partial L}{\partial \mathbf{b}_{2}} = \left[ -\frac{2}{N}(\mathbf{y} - \mathbf{\hat{y}}) \right] ^{\text{T}} \mathbf{1}.
$$
Agora o resultado terá a dimensão correta e temos uma multiplicação matricial válida. Teremos que repetir essa análise a cada derivada para verificar se a multiplicação matricial é possível e se a dimensão do resultado final é compatível com a derivada que estamos tomando.

Seguindo o fluxo de trabalho contrário, agora calcularemos a derivada de $L$ em relação a $\mathbf{W}_{2}$.
$$
\frac{\partial L}{\partial \mathbf{W}_{2}} = \frac{\partial L}{\partial \mathbf{\hat{y}}}\frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{W}_{2}}.
$$
Como
$$
\frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{W}_{2}} = \mathbf{z},
$$
temos que
$$
\frac{\partial L}{\partial \mathbf{W}_{2}} =  -\frac{2}{N}(\mathbf{y} - \mathbf{\hat{y}})\mathbf{z}.
$$
Vamos fazer as duas análises. Primeiro, vejamos qual a dimensão que nosso resultado deve ter: $\mathbf{W}_{2}$ tem dimensão $4\times 1$ e, portanto, nosso resultado deve ter essa dimensão. A multiplicação matricial acima é entre uma matriz $5\times 1$ e uma matriz $5\times 4$. Novamente, uma multiplicação inválida. Para que a multiplicação seja válida e o resultado tenha a dimensão desejada, devemos escrever que

$$
\frac{\partial L}{\partial \mathbf{W}_{2}} =  \mathbf{z}^{\text{T}}\left[-\frac{2}{N}(\mathbf{y} - \mathbf{\hat{y}})\right].
$$

Continuando, calcularemos a derivada de $L$ em relação a $\mathbf{b}_{1}$, dada por
$$
\frac{\partial L}{\partial \mathbf{b}_{1}} = \frac{\partial L}{\partial \mathbf{\hat{y}}}\frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}}\frac{\partial \mathbf{z}}{\partial \mathbf{H}_{1}}\frac{\partial \mathbf{H}_{1}}{\partial \mathbf{b}_{1}}.
$$
A expressão é ligeiramente maior, mas não é complexa. Vamos começar pela derivada de $\mathbf{\hat{y}}$ em relação a $\mathbf{z}$. Temos que ela será
$$
\frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}} = \mathbf{W}_{2}.
$$
A derivada de $\mathbf{z}$ em relação a $\mathbf{H}_{1}$ também é simples: se o elemento for negativo, então o resultado da função ReLU é zero, consequentemente, sua derivada também será. Se o elemento for positivo, então o resultado da função ReLU é o próprio elemento, cuja derivada em relação a ele mesmo será $1$. Portanto, a derivada da função ReLU nada mais é do que a função de Heaviside (ou função degrau), dada por
$$
\theta(x) =
\begin{cases}
0, \quad x < 0\\
1, \quad x> 0
\end{cases}.
$$
Assim, temos que
$$
\frac{\partial \mathbf{z}}{\partial \mathbf{H}_{1}} = \theta(\mathbf{H}_{1}),
$$
em que $\theta$ é aplicada elemento a elemento em $\mathbf{H}_{1}$ e, portanto, tem dimensão $5\times 4$.
Finalmente, a última derivada é, simplesmente,
$$
\frac{\partial \mathbf{H}_{1}}{\partial \mathbf{b}_{1}} = \mathbf{1},
$$
em que, agora, $\mathbf{1}$ tem dimensão $5\times 1$.

Vamos juntar tudo em uma expressão e analisar a dimensão do resultado e a ordem das operações.
$$
\frac{\partial L}{\partial \mathbf{b}_{1}} = \left[ -\frac{2}{N}(\mathbf{y} - \mathbf{\hat{y}})\right] \mathbf{W}_{2} \theta(\mathbf{H}_{1}) \mathbf{1}.
$$
A matriz $\mathbf{b}_{1}$ tem dimensão $1\times 4$, que é o nosso alvo. Vamos analisar a expressão acima substituindo os termos pelas suas dimensões:
$$
(5\times 1)(4\times 1)(5\times 4)(5\times 1)
$$
Tomamos a transposta da $(4\times 1)$ e realizamos a primeira multiplicação matricial, ficando com
$$
(5\times 1)(4\times 1)(5\times 4)(5\times 1) = (5\times 1)(1\times 4)(5\times 4)(5\times 1) = (5\times 4)(5\times 4)(5\times 1).
$$
A próxima multiplicação envolve o resultado dos dois primeiros termos (a derivada da função de perda e $\mathbf{W}_{2}$) e a função de Heaviside. Aqui há um detalhe importante: a função de ativação é aplicada a cada elemento da matriz $\mathbf{H}_{1}$ individualmente, isto é, não é feita nenhuma combinação dos resultados, há uma correspondência direta entre a entrada e saída da função de ativação. Isso é diferente de quando a informação passa por neurônios, onde são feitas combinações lineares e ocorre uma "mistura" devido às multiplicações matriciais. Então, quando estamos passando pela função de ativação, seja no caminho de ida (entrada para saída), seja no caminho de volta (da saída para a entrada), a ação da função de ativação atua **elemento a elemento**. Denotaremos essa operação por $\odot$. Portanto, a próxima operação é, na verdade,
$$
(5\times 4)\odot (5\times 4)(5\times 1),
$$
resultando em
$$
(5\times 4)(5\times 1).
$$
Tomando a transposta do segundo termo e trocando a ordem, ficamos com
$$
(1\times 5)(5\times 4) = (1\times 4),
$$
a dimensão desejada. Refazendo os passos com as matrizes, chegamos que
$$
\frac{\partial L}{\partial \mathbf{b}_{1}} = \mathbf{1}^{\text{T}}\left( \left\{ \left[ -\frac{2}{N}(\mathbf{y} - \mathbf{\hat{y}})\right] \mathbf{W}_{2}^{\text{T}} \right\}\odot \theta(\mathbf{H}_{1})\right).
$$

Finalmente, indo para a última derivada, de $L$ em relação a $\mathbf{W}_{1}$, temos que
$$
\frac{\partial L}{\partial \mathbf{W}_{1}} = \frac{\partial L}{\partial \mathbf{\hat{y}}}\frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}}\frac{\partial \mathbf{z}}{\partial \mathbf{H}_{1}}\frac{\partial \mathbf{H}_{1}}{\partial \mathbf{W}_{1}}.
$$
Para a nossa sorte, a maioria das derivadas já estão calculadas, faltando apenas a derivada de $\mathbf{H}_{1}$ em relação a $\mathbf{W}_{1}$:
$$
\frac{\partial \mathbf{H}_{1}}{\partial \mathbf{W}_{1}} = \mathbf{x}.
$$
Nosso resultado deve ter a dimensão de $\mathbf{W}_{1}$, que é $1\times 4$. Embora a última derivada seja diferente, todas as dimensões iguais as dimensões do caso anterior, ou seja, o resultado é análogo, mas para $\mathbf{x}$ no lugar de $\mathbf{1}$. Logo,
$$
\frac{\partial L}{\partial \mathbf{W}_{1}} = \mathbf{x}^{\text{T}}\left( \left\{ \left[ -\frac{2}{N}(\mathbf{y} - \mathbf{\hat{y}})\right] \mathbf{W}_{2}^{\text{T}} \right\}\odot \theta(\mathbf{H}_{1})\right).
$$

Com essas derivadas, podemos utilizar o algoritmo do gradiente descendente para atualiarmos os valores dos pesos e *bias* da rede neural a fim de melhorar a qualidade da previsão. Vamos implementar manualmente primeiro.

In [3]:
x = np.array([[1], [2], [3], [4], [5]]) # Dados de entrada
Y = 5 * x - 1 # Gerando os dados medidos

dim_n1 = (1, 4) # Definindo dimensões da primeira camada
dim_n2 = (4, 1) # Definindo dimensões da camada de saída

mlp = MLP(dim_n1, dim_n2) # Inicializando nosso MLP

y = mlp.forward(x) # Passando os dados de entrada pela rede
mse = mlp.MSE(Y, y) # Calculando o MSE dos dados previstos

print(f'Resultados da primeira iteração:\n')
print(f'Camada 1:\n\nW1 = {mlp.camada1.pesos}\nb1 = {mlp.camada1.bias}\n'+30*'-')
print(f'Camada de saída:\n\n W2 =\n{mlp.camada_saida.pesos}\nb2 = {mlp.camada_saida.bias}\n'+30*'-')
print(f'Saída:\n\n y =\n{y}\n\nMSE = {mse}')

Resultados da primeira iteração:

Camada 1:

W1 = [[ 0.43010115 -0.17095126 -0.24394639  0.47306877]]
b1 = [[ 0.64489207  1.155649   -2.05952565  0.36845887]]
------------------------------
Camada de saída:

 W2 =
[[ 0.22778657]
 [ 1.77181758]
 [-1.0164603 ]
 [ 0.25844521]]
b2 = [[-1.36829304]]
------------------------------
Saída:

 y =
[[0.83876954]
 [0.75610872]
 [0.6734479 ]
 [0.59078708]
 [0.50812625]]

MSE = 229.26387299973098


Agora, vamos implementar as derivadas e atualizar os valores dos pesos.

In [4]:
dLdy = (-2 / len(y)) * (Y - y) # Derivada de L em relação a y
dydb2 = np.ones((len(y), 1)) # Derivada de y em relação a b2
dydW2 = mlp.z # Derivada de y em relação a W2
dydz = mlp.camada_saida.pesos # Derivada de y em relação a z
dzdH1 = np.heaviside(mlp.H1, 0) # Derivada de z em relação a H1
dH1db1 = np.ones(((len(y), 1))) # Derivada de H1 em relação a b1
dH1dW1 = x # Derivada de H1 em relação a W1

dLdW1 = np.dot(dH1dW1.T, np.dot(dLdy, dydz.T) * dzdH1) # Derivada de L em relação a W1
dLdb1 = np.dot(dH1db1.T, np.dot(dLdy, dydz.T) * dzdH1) # Derivada de L em relação a b1
dLdW2 = np.dot(dydW2.T, dLdy) # Derivada de L em relação a W2
dLdb2 = np.dot(dLdy.T, dydb2) # Derivada de L em relação a b2

print('Valores das derivadas:\n')
print(f'dLdW1 = {dLdW1}\n')
print(f'dLdb1 = {dLdb1}\n')
print(f'dLdW2 =\n{dLdW2}\n')
print(f'dLdb2 = {dLdb2}')

# Atualizando os pesos com taxa de aprendizado eta
eta = 0.001

mlp.camada1.pesos -= eta * dLdW1
mlp.camada1.bias -= eta * dLdb1
mlp.camada_saida.pesos -= eta * dLdW2
mlp.camada_saida.bias -= eta * dLdb2

Valores das derivadas:

dLdW1 = [[ -22.84470469 -177.69550686    0.          -25.91945894]]

dLdb1 = [[ -6.07121909 -47.22443857   0.          -6.88836717]]

dLdW2 =
[[-60.32320036]
 [-13.65693922]
 [  0.        ]
 [-57.26461903]]

dLdb2 = [[-26.65310421]]


Depois de atualizar os pesos, fazemos o *forward propagation* de novo para obter uma nova previsão $\mathbf{\hat{y}}$.

In [5]:
y = mlp.forward(x) # Passando os dados de entrada pela rede
mse = mlp.MSE(Y, y) # Calculando o MSE dos dados previstos

print(f'Resultados da segunda iteração:\n')
print(f'Camada 1:\n\nW1 = {mlp.camada1.pesos}\nb1 = {mlp.camada1.bias}\n'+30*'-')
print(f'Camada de saída:\n\n W2 =\n{mlp.camada_saida.pesos}\nb2 = {mlp.camada_saida.bias}\n'+30*'-')
print(f'Saída:\n\n y =\n{y}\n\nMSE = {mse}')

Resultados da segunda iteração:

Camada 1:

W1 = [[ 0.45294585  0.00674425 -0.24394639  0.49898823]]
b1 = [[ 0.65096329  1.20287344 -2.05952565  0.37534724]]
------------------------------
Camada de saída:

 W2 =
[[ 0.28810977]
 [ 1.78547452]
 [-1.0164603 ]
 [ 0.31570983]]
b2 = [[-1.34163994]]
------------------------------
Saída:

 y =
[[1.41218494]
 [1.71226024]
 [2.01233553]
 [2.31241083]
 [2.61248613]]

MSE = 187.8826837643709


Como podemos ver, a função de perda diminuiu significativamente. Vamos atualizar os pesos mais uma vez fazer o *forward propagation*.

In [6]:
dLdy = (-2 / len(y)) * (Y - y) # Derivada de L em relação a y
dydb2 = np.ones((len(y), 1)) # Derivada de y em relação a b2
dydW2 = mlp.z # Derivada de y em relação a W2
dydz = mlp.camada_saida.pesos # Derivada de y em relação a z
dzdH1 = np.heaviside(mlp.H1, 0) # Derivada de z em relação a H1
dH1db1 = np.ones(((len(y), 1))) # Derivada de H1 em relação a b1
dH1dW1 = x # Derivada de H1 em relação a W1

dLdW1 = np.dot(dH1dW1.T, np.dot(dLdy, dydz.T) * dzdH1) # Derivada de L em relação a W1
dLdb1 = np.dot(dH1db1.T, np.dot(dLdy, dydz.T) * dzdH1) # Derivada de L em relação a b1
dLdW2 = np.dot(dydW2.T, dLdy) # Derivada de L em relação a W2
dLdb2 = np.dot(dLdy.T, dydb2) # Derivada de L em relação a b2

print('Valores das derivadas:\n')
print(f'dLdW1 = {dLdW1}\n')
print(f'dLdb1 = {dLdb1}\n')
print(f'dLdW2 =\n{dLdW2}\n')
print(f'dLdb2 = {dLdb2}\n\n')

# Atualizando os pesos com taxa de aprendizado eta
eta = 0.001

mlp.camada1.pesos -= eta * dLdW1
mlp.camada1.bias -= eta * dLdb1
mlp.camada_saida.pesos -= eta * dLdW2
mlp.camada_saida.bias -= eta * dLdb2

y = mlp.forward(x) # Passando os dados de entrada pela rede
mse = mlp.MSE(Y, y) # Calculando o MSE dos dados previstos

print(f'Resultados da terceira iteração:\n')
print(f'Camada 1:\n\nW1 = {mlp.camada1.pesos}\nb1 = {mlp.camada1.bias}\n'+30*'-')
print(f'Camada de saída:\n\n W2 =\n{mlp.camada_saida.pesos}\nb2 = {mlp.camada_saida.bias}\n'+30*'-')
print(f'Saída:\n\n y =\n{y}\n\nMSE = {mse}')

Valores das derivadas:

dLdW1 = [[ -26.13895611 -161.9883998     0.          -28.6429909 ]]

dLdb1 = [[ -6.90752642 -42.80733888   0.          -7.56924705]]

dLdW2 =
[[-56.70088169]
 [-29.45116307]
 [  0.        ]
 [-54.27012292]]

dLdb2 = [[-23.97532893]]


Resultados da terceira iteração:

Camada 1:

W1 = [[ 0.47908481  0.16873265 -0.24394639  0.52763122]]
b1 = [[ 0.65787081  1.24568078 -2.05952565  0.38291649]]
------------------------------
Camada de saída:

 W2 =
[[ 0.34481065]
 [ 1.81492568]
 [-1.0164603 ]
 [ 0.36997995]]
b2 = [[-1.31766461]]
------------------------------
Saída:

 y =
[[1.97830946]
 [2.64495319]
 [3.31159693]
 [3.97824066]
 [4.6448844 ]]

MSE = 151.7979132838791


Como vemos, a função de perda está diminuindo. Em poucas iterações, caiu para quase metade do valor inicial.

## 0.3.2 *Backpropagation*

O conceito de atualizar os parâmetros de trás para frente é chamado de *backpropagation* e, como vimos, é fundamental para o aprendizado da rede. Na verdade, é justamente *isso* que possibilita a rede a aprender um padrão e ser capaz de prever bem algum valor. Vamos adicionar um método `.backward()` na classe `MLP` que calcula isso automaticamente.

In [7]:
class MLP:
    ''' Define um multilayer perceptron de duas camadas.
    Atributos:
        camada1: a primeira camada do MLP.
        camada_saida: a camada de saída do MLP.

    Parâmetros:
        dim_n1: lista ou tupla contendo as dimensões da primeira camada na forma (n_entrada, n_saida).
        dim_n1: lista ou tupla contendo as dimensões da camada de saída na forma (n_entrada, n_saida).
    '''

    def __init__(self, dim_n1, dim_n2):
        self.camada1 = Camada(dim_n1[0], dim_n1[1])
        self.camada_saida = Camada(dim_n2[0], dim_n2[1])
        self.H1 = 0.0 # Atributo para salvarmos a saída da camada 1
        self.z = 0.0 # Atributo para salvarmos a saída da ReLU

    # Definindo a função ReLU para ser usada como função de ativação
    def relu(self, entrada):
        zeros = np.zeros_like(entrada)
        self.z = np.maximum(zeros, entrada)
        return self.z

    # Definindo o método para executar o forward propagation pela rede
    def forward(self, entrada):
        H1 = self.camada1.forward(entrada) # Passa para a camada 1
        self.H1 = H1
        z = self.relu(H1) # Passa a saída da camada 1 pela função de ativação
        y = self.camada_saida.forward(z) # Passa a saída da função de ativação para a camada de saída
        return y

    # Definindo a função de perda MSE. Parâmetros: Y: dados observados; y: dados previstos
    def MSE(self, Y, y):
        erro = Y - y
        return np.mean(erro**2)

    # Definindo o método para executar o backpropagation pela rede
    def backward(self, Y, y, x, eta):
        dLdy = (-2 / len(y)) * (Y - y) # Derivada de L em relação a y
        dydb2 = np.ones((len(y), 1)) # Derivada de y em relação a b2
        dydW2 = self.z # Derivada de y em relação a W2
        dydz = self.camada_saida.pesos # Derivada de y em relação a z
        dzdH1 = np.heaviside(self.H1, 0) # Derivada de z em relação a H1
        dH1db1 = np.ones(((len(y), 1))) # Derivada de H1 em relação a b1
        dH1dW1 = x # Derivada de H1 em relação a W1
        
        dLdW1 = np.dot(dH1dW1.T, np.dot(dLdy, dydz.T) * dzdH1) # Derivada de L em relação a W1
        dLdb1 = np.dot(dH1db1.T, np.dot(dLdy, dydz.T) * dzdH1) # Derivada de L em relação a b1
        dLdW2 = np.dot(dydW2.T, dLdy) # Derivada de L em relação a W2
        dLdb2 = np.dot(dLdy.T, dydb2) # Derivada de L em relação a b2
        
        # Atualizando os pesos com taxa de aprendizado eta        
        self.camada1.pesos -= eta * dLdW1
        self.camada1.bias -= eta * dLdb1
        self.camada_saida.pesos -= eta * dLdW2
        self.camada_saida.bias -= eta * dLdb2
        

Vamos reiniciar a rede e rodar a análise de novo.

In [8]:
x = np.array([[1], [2], [3], [4], [5]])
Y = 5 * x - 1

dim_n1 = (1, 4)
dim_n2 = (4, 1)

mlp = MLP(dim_n1, dim_n2)

y = mlp.forward(x)
mse = mlp.MSE(Y, y)
mlp.backward(Y, y, x, eta=0.001) # Executa o backpropagation

print(f'A saída é y =\n{y}')

A saída é y =
[[0.7081872 ]
 [2.74330781]
 [4.19325915]
 [5.64321048]
 [7.09316182]]


Vamos implementar esse mecanismo em um *loop* `while` semelhante à etapa 0.1.

In [9]:
parada = 10e-4
err = 1
p = 250
i = 1
eta = 0.001

while err > parada:
    y = mlp.forward(x)
    if i == 1:
        mse = mlp.MSE(Y, y)
        new_mse = 0
    else:
        new_mse = mlp.MSE(Y, y)

    mlp.backward(Y, y, x, eta)

    if (i+1)%1000 == 0 or i==0: # Imprime os resultados a cada 1000 iterações
        print(30*'-'+f'\nIteração {i+1}\n')
        print(f'MSE = {mse}\n')
        print(f'y =\n{y}')

    # Calcula a diferença entre os MSE a cada n iterações
    if (i)%p == 0 and i != 0: # Evita que o programa entre aqui na primeira iteração (em que err = 0 e o programa pararia)
        err = mse - new_mse
        mse = new_mse

    i += 1

    if err < parada: # Imprime o último resultado
        print(30*'-'+f'\nIteração {i+1}\n')
        print(f'MSE = {mse}\n')
        print(f'y =\n{y}')

------------------------------
Iteração 1000

MSE = 0.00355818569170335

y =
[[ 3.94804075]
 [ 8.96735551]
 [13.98667028]
 [19.00598504]
 [24.0252998 ]]
------------------------------
Iteração 1252

MSE = 0.00023877981553523223

y =
[[ 3.97358464]
 [ 8.98340542]
 [13.9932262 ]
 [19.00304699]
 [24.01286777]]


A nossa rede precisou de quase $12000$ iterações para satisfazer o critério de parada. Para comparar, o algoritmo do gradiente descendente da etapa 0.1 precisou de cerca de $22500$ iterações, um número expressivamente maior. Vale lembrar que estamos aproximando uma função simples, para tarefas mais complexas, esse ganho de velocidade é muito mais significativo. Vamos tentar prever um valor, agora que nossa rede está treinada. Para isso, basta executarmos o `forward` para o valor específico.

In [10]:
a = np.array([[2.7]])
b = mlp.forward(a)
print(f'O valor exato é {5 * a[0][0] - 1}, a rede neural previu o valor {b[0][0]:.4f}.')

O valor exato é 12.5, a rede neural previu o valor 12.4903.


A rede neural fez uma previsão *ótima*, com uma boa precisão. Refletindo o bom treinamento e o baixíssimo MSE obtido no treino.

**No entanto**, precisamos lembrar que, no geral, uma rede neural só consegue prever adequadamente o resultado de valores que estão *dentro* do intervalo dos dados, isto é, se quisermos prever a saída de uma entrada $x$, é necessário que $d_{\text{min.}} < x < d_{\text{max.}}$, em que $d_{\text{min.}}$ e $d_{\text{max.}}$ representam o menor e maior valor dos dados de entrada de treinamento. Para relações lineares, é possível com a função ReLU extrapolar os dado, mas, para funções não-lineares, a extrapolação não é muito confiável.